In [ ]:
# === Pacotes necessários / Required packages ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.spatial.distance import mahalanobis

# Sem classe indexada / No indexed class

In [ ]:
# ===== 1. Ler os dados .XLSX / Read .XLSX data =====
df = pd.read_excel("NAME.xlsx")
print(df.head())

In [ ]:
# === Ler os dados .CSV / Read .CSV data ===
df = pd.read_csv('NAME.csv', sep=',')  # especifica o separador / specifies the separator
print(df.head())

In [ ]:
# Selecionar apenas colunas numéricas / Select only numeric columns
df = df.select_dtypes(include=[np.number])

print(df.head())

In [ ]:
# === Normalizar e padronizar / Normalize and standardize ===
scaler_std = StandardScaler()
df_pad = pd.DataFrame(scaler_std.fit_transform(df), columns=df.columns)

df_norm = df.copy()
for col in df.columns:
    x = df[col]
    df_norm[col] = (x - x.min()) / (x.max() - x.min())

In [ ]:
# === Plotagem de histogramas / Histogram plotting ===
for col in df.columns:
    x = df[col].dropna()

    plt.figure()
    plt.hist(x, bins=20, density=True, color="blue", alpha=0.7)

    media = x.mean()
    desvio = x.std()

    xx = np.linspace(x.min(), x.max(), 100)
    yy = (1/(desvio*np.sqrt(2*np.pi))) * np.exp(-0.5*((xx-media)/desvio)**2)
    plt.plot(xx, yy, color="red", linewidth=2)

    plt.title(f"Hist of {col}")
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.show()


In [ ]:
# === Gráficos de dispersão / Scatter Plots ===
for col in df.columns:
    x = df[col]
    media = x.mean()
    desvio = x.std()
    limite_sup = media + 3*desvio
    limite_inf = media - 3*desvio

    ylim_inf = min(limite_inf, x.min())
    ylim_sup = max(limite_sup, x.max())

    plt.figure()
    plt.scatter(range(len(x)), x, color="blue", s=20)
    plt.axhline(media, color="red", linewidth=2)
    plt.axhline(limite_sup, color="red", linewidth=2)
    plt.axhline(limite_inf, color="red", linewidth=2)
    plt.ylim([ylim_inf, ylim_sup])

    plt.title(f"Dispersion of {col}")
    plt.xlabel("Observation index")
    plt.ylabel(col)
    plt.show()

In [ ]:
# === Limpeza de outliers univariados / Univariate outlier cleaning ===
df_cleaned = df.copy()
for col in df.columns:
    media = df_cleaned[col].mean()
    desvio = df_cleaned[col].std()
    limite_sup = media + 3*desvio
    limite_inf = media - 3*desvio
    df_cleaned = df_cleaned[(df_cleaned[col] >= limite_inf) & (df_cleaned[col] <= limite_sup)]

In [ ]:
df_univariate_cleaned = df_cleaned.copy()

# === Salvar em Excel os dados limpos univariados / Save univariate cleaned data to Excel ===
df_univariate_cleaned.to_excel("Dados_limpos_univariados.xlsx", index=False)

# === Salvar em CSV os dados limpos univariados / Save univariate cleaned data to CSV ===
df_univariate_cleaned.to_csv("Dados_limpos_univariados.csv", index=False)

In [ ]:
# === Outliers Multivariados (Distância de Mahalanobis) / Multivariate Outliers (Mahalanobis Distance) ===

# Calcular médias e matriz de covariância dos dados limpos / Calculate means and covariance matrix of cleaned data
medias_sdi = df_cleaned.mean().values
cov_sdi = np.cov(df_cleaned.values, rowvar=False)
inv_cov_sdi = np.linalg.inv(cov_sdi)

# Calcular distância de Mahalanobis para cada observação / Calculate Mahalanobis distance for each observation
sdi = []
for i in range(len(df_cleaned)):
    x = df_cleaned.iloc[i].values
    dist = mahalanobis(x, medias_sdi, inv_cov_sdi)
    sdi.append(dist)
sdi = np.array([mahalanobis(x, medias_sdi, inv_cov_sdi)**2 for x in df_cleaned.values])

# Estatísticas da distância / Distance statistics
medias_sdi_val = sdi.mean()
desvpad_sdi = sdi.std()
cortesup = medias_sdi_val + 3*desvpad_sdi
corteinf = medias_sdi_val - 3*desvpad_sdi

# Plotar as distâncias / Plot the distances
plt.figure()
plt.scatter(range(len(sdi)), sdi, c="blue", s=20)
plt.axhline(medias_sdi_val, color="red", linewidth=3)
plt.axhline(cortesup, color="red", linestyle="--", linewidth=2)
plt.axhline(corteinf, color="red", linestyle="--", linewidth=2)
plt.title("Statistical Distances - Mahalanobis")
plt.xlabel("Observation")
plt.ylabel("Statistical Distances")
plt.show()

# Remover outliers multivariados / Remove multivariate outliers
df_cleaned = df_cleaned[sdi <= cortesup]

In [ ]:
# === Visualização pós-limpeza / Post-cleaning view ===
for col in df_cleaned.columns:
    x = df_cleaned[col]
    media = x.mean()
    desvio = x.std()
    limite_sup = media + 3*desvio
    limite_inf = media - 3*desvio

    ylim_inf = min(limite_inf, x.min())
    ylim_sup = max(limite_sup, x.max())

    plt.figure()
    plt.scatter(range(len(x)), x, color="blue", s=20)
    plt.axhline(media, color="red", linewidth=2)
    plt.axhline(limite_sup, color="red", linewidth=2)
    plt.axhline(limite_inf, color="red", linewidth=2)
    plt.ylim([ylim_inf, ylim_sup])

    plt.title(f"Dispersion of {col} (clean data)")
    plt.xlabel("Observation index")
    plt.ylabel(col)
    plt.show()


In [ ]:
# === Salvar em Excel / Save to Excel ===
df_cleaned.to_excel("Dados_limpos.xlsx", index=False)


In [ ]:
# === Salvar em CSV / Save to CSV ===
df_cleaned.to_csv("Dados_limpos.csv", index=False)

# Com classe indexada / With indexed class

In [ ]:
# ===== 1. Ler os dados .XLSX / Read .XLSX data =====
df = pd.read_excel("NAME.xlsx")
print(df.head())

In [ ]:
# === Ler os dados .CSV / Read .CSV data ===
df = pd.read_csv('NAME.csv', sep=',')  # especifica o separador / specifies the separator
print(df.head())

In [ ]:
# Separar classe e dados numéricos / Separate class and numeric data
classe = df.iloc[:, 0]
X = df.iloc[:, 1:].select_dtypes(include=[np.number])

print(X.head())

In [ ]:
# === Normalizar e padronizar / Normalize and standardize ===
scaler_std = StandardScaler()
df_pad = pd.DataFrame(scaler_std.fit_transform(X), columns=X.columns)

df_norm = X.copy()
for col in X.columns:
    x = X[col]
    df_norm[col] = (x - x.min()) / (x.max() - x.min())

In [ ]:
# === Plotagem de histogramas / Histogram plotting ===
for col in X.columns:
    x = X[col].dropna()
    plt.figure()
    plt.hist(x, bins=20, density=True, color="blue", alpha=0.7)
    media = x.mean()
    desvio = x.std()
    xx = np.linspace(x.min(), x.max(), 100)
    yy = (1/(desvio*np.sqrt(2*np.pi))) * np.exp(-0.5*((xx-media)/desvio)**2)
    plt.plot(xx, yy, color="red", linewidth=2)
    plt.title(f"Hist of {col}")
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.show()

In [ ]:
# === Gráficos de dispersão / Scatter Plots ===
for col in X.columns:
    x = X[col]
    media = x.mean()
    desvio = x.std()
    limite_sup = media + 3*desvio
    limite_inf = media - 3*desvio
    ylim_inf = min(limite_inf, x.min())
    ylim_sup = max(limite_sup, x.max())
    plt.figure()
    plt.scatter(range(len(x)), x, color="blue", s=20)
    plt.axhline(media, color="red", linewidth=2)
    plt.axhline(limite_sup, color="red", linewidth=2)
    plt.axhline(limite_inf, color="red", linewidth=2)
    plt.ylim([ylim_inf, ylim_sup])
    plt.title(f"Dispersion of {col}")
    plt.xlabel("Observation index")
    plt.ylabel(col)
    plt.show()

In [ ]:
# === Limpeza de outliers univariados / Univariate outlier cleaning ===
df_cleaned = X.copy()
for col in X.columns:
    media = df_cleaned[col].mean()
    desvio = df_cleaned[col].std()
    limite_sup = media + 3*desvio
    limite_inf = media - 3*desvio
    df_cleaned = df_cleaned[(df_cleaned[col] >= limite_inf) & (df_cleaned[col] <= limite_sup)]

In [ ]:
#Salvar univariados limpos / save univariated cleaning data

df_univariate_cleaned_with_class = df_cleaned.copy()

# Adicionar coluna de classe de volta / Add class column back
df_univariate_cleaned_with_class['Classe'] = classe.loc[df_univariate_cleaned_with_class.index]

# Reorder columns to put 'Classe' as the first column
cols = ['Classe'] + [col for col in df_univariate_cleaned_with_class.columns if col != 'Classe']
df_univariate_cleaned_with_class = df_univariate_cleaned_with_class[cols]

# === Salvar em Excel os dados limpos univariados com classe / Save univariate cleaned data with class to Excel ===
df_univariate_cleaned_with_class.to_excel("Dados_limpos_univariados_com_classe.xlsx", index=False)

# === Salvar em CSV os dados limpos univariados com classe / Save univariate cleaned data with class to CSV ===
df_univariate_cleaned_with_class.to_csv("Dados_limpos_univariados_com_classe.csv", index=False)

print(df_univariate_cleaned_with_class.head())

 Multivariate Outliers: general data

In [ ]:
# === Outliers Multivariados (Distância de Mahalanobis) / Multivariate Outliers (Mahalanobis Distance) ===

# Calcular médias e matriz de covariância dos dados limpos / Calculate means and covariance matrix of cleaned data
medias_sdi = df_cleaned.mean().values
cov_sdi = np.cov(df_cleaned.values, rowvar=False)
inv_cov_sdi = np.linalg.inv(cov_sdi)

# Calcular distância de Mahalanobis para cada observação / Calculate Mahalanobis distance for each observation
sdi = []
for i in range(len(df_cleaned)):
    x = df_cleaned.iloc[i].values
    dist = mahalanobis(x, medias_sdi, inv_cov_sdi)
    sdi.append(dist)
sdi = np.array([mahalanobis(x, medias_sdi, inv_cov_sdi)**2 for x in df_cleaned.values])

# Estatísticas da distância / Distance statistics
medias_sdi_val = sdi.mean()
desvpad_sdi = sdi.std()
cortesup = medias_sdi_val + 3*desvpad_sdi
corteinf = medias_sdi_val - 3*desvpad_sdi

# Plotar as distâncias / Plot the distances
plt.figure()
plt.scatter(range(len(sdi)), sdi, c="blue", s=20)
plt.axhline(medias_sdi_val, color="red", linewidth=3)
plt.axhline(cortesup, color="red", linestyle="--", linewidth=2)
plt.axhline(corteinf, color="red", linestyle="--", linewidth=2)
plt.title("Statistical Distances (Mahalanobis)")
plt.xlabel("Observation")
plt.ylabel("Statistical distance")
plt.show()

# Remover outliers multivariados / Remove multivariate outliers
df_cleaned = df_cleaned[sdi <= cortesup]

# === Visualização pós-limpeza / Post-cleaning view ===
if df_cleaned.empty:
    print("O DataFrame df_cleaned está vazio. Não há dados para plotar.")
else:
    for col in df_cleaned.columns:
        x = df_cleaned[col]
        media = x.mean()
        desvio = x.std()
        limite_sup = media + 3*desvio
        limite_inf = media - 3*desvio

        # Garante que não usará NaN se a série contiver NaNs ou estiver vazia
        ylim_inf = min(limite_inf, x.min()) if pd.notna(limite_inf) and pd.notna(x.min()) else 0
        ylim_sup = max(limite_sup, x.max()) if pd.notna(limite_sup) and pd.notna(x.max()) else 1

        plt.figure()
        plt.scatter(range(len(x)), x, color="blue", s=20)
        plt.axhline(media, color="red", linewidth=2)
        plt.axhline(limite_sup, color="red", linewidth=2)
        plt.axhline(limite_inf, color="red", linewidth=2)
        plt.ylim([ylim_inf, ylim_sup])
        plt.title(f"Dispersion of {col} (Clean Data)")
        plt.xlabel("Observation index")
        plt.ylabel(col)
        plt.show()

 Multivariate Outliers: Spectra

In [ ]:
# === Avaliação de Parâmetros do QC Multivariado (PCA + Mahalanobis) / Multivariate QC Parameter Evaluation (PCA + Mahalanobis) ===

from sklearn.decomposition import PCA
from scipy.stats import chi2

# Parâmetros a avaliar / Parameters to evaluate
n_components_list = [2, 3, 4, 5, 6, 7, 8]
alpha_sdi_list = [0.95, 0.975, 0.99]
k_odi_list = [2.5, 3.0, 3.5]

# DataFrame para armazenar os resultados / DataFrame to store results
pca_qc_results = []

# Avaliar todas as combinações / Evaluate all combinations
for n_components_pca in n_components_list:

    # Aplicar PCA / Apply PCA
    pca_sdi = PCA(n_components=n_components_pca, svd_solver="full")
    scores_sdi = pca_sdi.fit_transform(df_cleaned.values)

    # Reconstrução a partir dos componentes principais / Reconstruction from principal components
    X_reconstructed_sdi = pca_sdi.inverse_transform(scores_sdi)

    # Distância ortogonal ao subespaço PCA / Orthogonal distance to PCA subspace
    odi = np.linalg.norm(df_cleaned.values - X_reconstructed_sdi, axis=1)

    # Média e matriz de covariância dos scores / Mean and covariance matrix of PCA scores
    medias_sdi = scores_sdi.mean(axis=0)
    cov_sdi = np.cov(scores_sdi, rowvar=False)
    inv_cov_sdi = np.linalg.inv(cov_sdi)

    # Calcular distância de Mahalanobis para cada observação / Calculate Mahalanobis distance for each observation
    sdi = []
    for i in range(len(scores_sdi)):
        x = scores_sdi[i]
        dist = mahalanobis(x, medias_sdi, inv_cov_sdi)
        sdi.append(dist)

    sdi = np.array(sdi)
    sdi_squared = sdi ** 2

    # Variância explicada / Explained variance
    variancia_explicada = np.sum(pca_sdi.explained_variance_ratio_)

    for alpha_sdi in alpha_sdi_list:

        # Limiar de Mahalanobis / Mahalanobis threshold
        cortesup_sdi = np.sqrt(chi2.ppf(alpha_sdi, df=n_components_pca))
        outliers_sdi = sdi_squared > cortesup_sdi ** 2

        for k_odi in k_odi_list:

            # Limiar da distância ortogonal / Orthogonal distance threshold
            mediana_odi = np.median(odi)
            mad_odi = np.median(np.abs(odi - mediana_odi))
            cortesup_odi = mediana_odi + k_odi * 1.4826 * mad_odi
            outliers_odi = odi > cortesup_odi

            # Outliers multivariados / Multivariate outliers
            outliers_multivariate = outliers_sdi | outliers_odi

            # Contagem por classe / Count by class
            contagem_classe = classe.loc[df_cleaned.index[outliers_multivariate]].value_counts()

            pca_qc_results.append({
                "n_components_pca": n_components_pca,
                "alpha_sdi": alpha_sdi,
                "k_odi": k_odi,
                "variance_explained": variancia_explicada,
                "n_outliers_sdi": outliers_sdi.sum(),
                "n_outliers_odi": outliers_odi.sum(),
                "n_outliers_total": outliers_multivariate.sum(),
                "percent_outliers": 100 * outliers_multivariate.sum() / len(df_cleaned),
                "outliers_by_class": str(contagem_classe.to_dict())
            })

# Converter resultados em DataFrame / Convert results to DataFrame
pca_qc_results = pd.DataFrame(pca_qc_results)

# Ordenar por número de outliers / Sort by number of outliers
pca_qc_results = pca_qc_results.sort_values(
    by="n_outliers_total",
    ascending=True
).reset_index(drop=True)

# Visualizar resultados / Display results
pca_qc_results

In [ ]:
# === Visualizar variância explicada / Visualize explained variance ===

plt.figure(figsize=(8, 4))

plt.plot(
    range(1, len(pca_sdi.explained_variance_ratio_) + 1),
    pca_sdi.explained_variance_ratio_,
    marker="o",
    label="Individual"
)

plt.plot(
    range(1, len(pca_sdi.explained_variance_ratio_) + 1),
    np.cumsum(pca_sdi.explained_variance_ratio_),
    marker="o",
    label="Cumulative"
)

plt.axhline(0.95, color="red", linestyle="--", linewidth=2)
plt.title("PCA Explained Variance")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.legend()
plt.show()

In [ ]:
# === Escolher parâmetros para inspeção / Select parameters for inspection ===

n_components_pca = 2
alpha_sdi = 0.9
k_odi = 2.5

# Aplicar PCA / Apply PCA
pca_sdi = PCA(n_components=n_components_pca, svd_solver="full")
scores_sdi = pca_sdi.fit_transform(df_cleaned.values)

# Reconstrução / Reconstruction
X_reconstructed_sdi = pca_sdi.inverse_transform(scores_sdi)

# Distâncias / Distances
odi = np.linalg.norm(df_cleaned.values - X_reconstructed_sdi, axis=1)

medias_sdi = scores_sdi.mean(axis=0)
cov_sdi = np.cov(scores_sdi, rowvar=False)
inv_cov_sdi = np.linalg.inv(cov_sdi)

sdi = np.array([
    mahalanobis(x, medias_sdi, inv_cov_sdi)
    for x in scores_sdi
])

sdi_squared = sdi ** 2

# Limiares / Thresholds
cortesup_sdi = np.sqrt(chi2.ppf(alpha_sdi, df=n_components_pca))

mediana_odi = np.median(odi)
mad_odi = np.median(np.abs(odi - mediana_odi))
cortesup_odi = mediana_odi + k_odi * 1.4826 * mad_odi

# Outliers / Outliers
outliers_sdi = sdi_squared > cortesup_sdi ** 2
outliers_odi = odi > cortesup_odi
outliers_multivariate = outliers_sdi | outliers_odi

# Plotar mapa de outliers / Plot outlier map
plt.figure(figsize=(8, 5))

plt.scatter(
    sdi_squared[~outliers_multivariate],
    odi[~outliers_multivariate],
    c="blue",
    s=20,
    label="Retained spectra"
)

plt.scatter(
    sdi_squared[outliers_multivariate],
    odi[outliers_multivariate],
    c="red",
    s=35,
    label="Multivariate outliers"
)

plt.axvline(
    cortesup_sdi ** 2,
    color="black",
    linestyle="--",
    linewidth=2,
    label="Score distance threshold"
)

plt.axhline(
    cortesup_odi,
    color="black",
    linestyle=":",
    linewidth=2,
    label="Orthogonal distance threshold"
)

plt.title("PCA Multivariate Outlier Map")
plt.xlabel("Squared Score Distance (Mahalanobis)")
plt.ylabel("Orthogonal Distance")
plt.legend()
plt.show()

In [ ]:
# === Outliers Multivariados (PCA + Mahalanobis Distance) / Multivariate Outliers (PCA + Mahalanobis Distance) ===

# =========================================================
# PARÂMETROS AJUSTÁVEIS / ADJUSTABLE PARAMETERS
# =========================================================

# Número de componentes principais / Number of principal components
n_components_pca = 2

# Nível de significância para o limiar da distância de Mahalanobis / Significance level for the Mahalanobis distance threshold
# 0.95 = mais restritivo | 0.975 = padrão | 0.99 = menos restritivo
alpha_sdi = 0.9

# Fator do limiar da distância ortogonal / Orthogonal distance threshold factor
# 2.5 = mais restritivo | 3.0 = padrão | 3.5 ou 4.0 = menos restritivo
k_odi = 2

# =========================================================
# FIM DOS PARÂMETROS AJUSTÁVEIS / END OF ADJUSTABLE PARAMETERS
# =========================================================


# Aplicar PCA nos dados limpos / Apply PCA to cleaned data
pca_sdi = PCA(n_components=n_components_pca, svd_solver="full")
scores_sdi = pca_sdi.fit_transform(df_cleaned.values)

# Reconstrução a partir dos componentes principais / Reconstruction from principal components
X_reconstructed_sdi = pca_sdi.inverse_transform(scores_sdi)

# Distância ortogonal ao subespaço PCA / Orthogonal distance to PCA subspace
odi = np.linalg.norm(df_cleaned.values - X_reconstructed_sdi, axis=1)

# Média e matriz de covariância dos scores / Mean and covariance matrix of PCA scores
medias_sdi = scores_sdi.mean(axis=0)
cov_sdi = np.cov(scores_sdi, rowvar=False)
inv_cov_sdi = np.linalg.inv(cov_sdi)

# Calcular distância de Mahalanobis para cada observação / Calculate Mahalanobis distance for each observation
sdi = []
for i in range(len(scores_sdi)):
    x = scores_sdi[i]
    dist = mahalanobis(x, medias_sdi, inv_cov_sdi)
    sdi.append(dist)

sdi = np.array(sdi)
sdi_squared = sdi ** 2

# Limiar de Mahalanobis baseado em qui-quadrado / Mahalanobis threshold based on chi-square distribution
cortesup_sdi = np.sqrt(chi2.ppf(alpha_sdi, df=n_components_pca))

# Estatísticas da distância ortogonal / Orthogonal distance statistics
mediana_odi = np.median(odi)
mad_odi = np.median(np.abs(odi - mediana_odi))
cortesup_odi = mediana_odi + k_odi * 1.4826 * mad_odi

# Identificar outliers por score distance / Identify score distance outliers
outliers_sdi = sdi_squared > cortesup_sdi ** 2

# Identificar outliers por orthogonal distance / Identify orthogonal distance outliers
outliers_odi = odi > cortesup_odi

# Identificar outliers multivariados / Identify multivariate outliers
outliers_multivariate = outliers_sdi | outliers_odi

# Plotar mapa de outliers / Plot outlier map
plt.figure(figsize=(8, 5))

plt.scatter(
    sdi_squared[~outliers_multivariate],
    odi[~outliers_multivariate],
    c="blue",
    s=20,
    label="Retained spectra"
)

plt.scatter(
    sdi_squared[outliers_multivariate],
    odi[outliers_multivariate],
    c="red",
    s=35,
    label="Multivariate outliers"
)

plt.axvline(
    cortesup_sdi ** 2,
    color="black",
    linestyle="--",
    linewidth=2,
    label="Score distance threshold"
)

plt.axhline(
    cortesup_odi,
    color="black",
    linestyle=":",
    linewidth=2,
    label="Orthogonal distance threshold"
)

plt.title("PCA Multivariate Outlier Map")
plt.xlabel("Squared Score Distance (Mahalanobis)")
plt.ylabel("Orthogonal Distance")
plt.legend()
plt.show()

# Relatório dos outliers / Outlier report
print("Número de componentes principais:", n_components_pca)
print("Variância explicada pelos PCs:", np.sum(pca_sdi.explained_variance_ratio_))
print("Limiar do score distance:", cortesup_sdi ** 2)
print("Limiar da orthogonal distance:", cortesup_odi)
print("Outliers por score distance:", outliers_sdi.sum())
print("Outliers por orthogonal distance:", outliers_odi.sum())
print("Total de outliers multivariados:", outliers_multivariate.sum())
print("Espectros restantes antes da remoção:", len(df_cleaned) - outliers_multivariate.sum())

# Remover outliers multivariados / Remove multivariate outliers
df_cleaned = df_cleaned[~outliers_multivariate]

print("Espectros restantes após a remoção:", len(df_cleaned))

Continue

In [ ]:
# === Adicionar coluna de classe de volta / Add class column back ===
df_cleaned["Classe"] = classe.loc[df_cleaned.index]

# Reorder columns to put 'Classe' as the first column
cols = ['Classe'] + [col for col in df_cleaned.columns if col != 'Classe']
df_cleaned = df_cleaned[cols]

In [ ]:
# === Salvar em Excel / Save to Excel ===
df_cleaned.to_excel("Dados_limpos.xlsx", index=False)

In [ ]:
# === Salvar em CSV / Save to CSV ===
df_cleaned.to_csv("Dados_limpos.csv", index=False)